# Tarea 2: Procesamiento de Texto con Python
### Integrantes:
* Sebastian Vargas
* Sebastian Leon
* Rodrigo Ramirez

**Asignatura:** Taller de Python para Ciencia de Datos

# Análisis Exploratorio de Datos (EDA)
Este notebook analiza el dataset crudo `global_freelancers_raw.csv`.
Las transformaciones y limpieza se realizan posteriormente en `02_preparation.ipynb`, aquí nos enfocamos solo en la exploración.


# 1. Configuración del Entorno y Carga de Datos Crudos

En esta primera sección configuramos el espacio de trabajo. Dado que el proyecto sigue una estructura modular profesional, realizamos las siguientes acciones iniciales:

* **Manipulación de Rutas Dinámicas:** Utilizamos la librería nativa `pathlib` para asegurar que el proyecto sea portable y funcione en cualquier sistema operativo sin romper las rutas.
* **Inyección en el Path del Sistema (`sys.path`):** Añadimos el directorio raíz del proyecto al entorno de Python para habilitar la importación directa de nuestros módulos personalizados localizados en la carpeta `src/`.
* **Configuración de Pandas:** Ajustamos las opciones de visualización para poder inspeccionar todas las columnas sin que el framework las oculte o las mutile en el flujo de trabajo.


In [ ]:
import pandas as pd
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))

data_path = Path.cwd().parent / 'data' / 'raw' / 'global_freelancers_raw.csv'
df = pd.read_csv(data_path)
df.head()

## 2. Diagnóstico de Dimensiones y Datos Faltantes (Nulos)

Para comenzar con el Análisis Exploratorio de Datos (EDA), realizamos una auditoría cuantitativa sobre el archivo original. El objetivo de esta celda es doble:

* Dimenciones de la tabla (`df.shape`): Determinamos la magnitud del dataset mediante el conteo total de observaciones (filas) y variables (columnas). Esto establece la línea base antes de cualquier proceso de filtrado o eliminación.
* conteo de nulls (`df.isnull().sum()`):** Identificamos y cuantificamos la presencia de valores nulos ($NaN$) en cada una de las columnas. 

Este paso es crítico, ya que nos permite diseñar la estrategia de limpieza que aplicaremos en el siguiente notebook, asegurando que las funciones estadísticas no se sesguen por culpa de registros incompletos.

In [ ]:
print(f"Dimensiones del dataset crudo: {df.shape[0]} filas y {df.shape[1]} columnas.\n")
print("Conteo de valores nulos por columna:")
print(df.isnull().sum())

## 3. Inspección de Consistencia en Variables Categóricas y Booleanas

En esta celda realizamos una revisión de los valores únicos en las columnas `gender` e `is_active` utilizando la función `.unique()`. El propósito de esta inspección es identificar la consistencia de los datos cualitativos:

* Control de Formato y Errores de Tipeo

In [ ]:
print("Valores únicos en Gender:", df['gender'].unique())
print("Valores únicos en Active:", df['is_active'].unique())

## 4. Resumen Estadístico Descriptivo Global

Ejecutamos `df.info()` y `df.describe(include='all').T` para realizar una auditoría estadística multivariable. Al transponer la matriz e incluir todas las variables, obtenemos métricas de tendencia central, dispersión y frecuencias en un formato scannable. El objetivo de esta celda es: lograr el analisis correcto para tener un marco de referencia en el cual trabajar los datos.

In [ ]:
df.info()


Obviando la cantidad de nulls que existen en la tabla, uno de los errores mas criticos de la base de datos para nuestro analisis:

Tipo de Dato Erróneo (Números leídos como Texto): hourly_rate (USD) es de tipo str (texto): Debería ser un número decimal (float64). Esto te confirma que los precios vienen sucios con caracteres extraños (como los signos $ o las letras USD) que obligaron a Pandas a leer la columna como texto. No puedes calcular promedios ni mínimos hasta limpiarla.

#### `df.describe(include='all').T`
* **Analizar la Distribución Numérica:** Evaluar promedios (`mean`), valores mínimos (`min`), máximos (`max`) y los cuartiles clave ($25\%$, $50\%$, $75\%$) para entender el comportamiento real de los freelancers (edad, experiencia y calificaciones).
* **Auditar la Variabilidad Categórica:** Examinar la cantidad de valores únicos (`unique`) y las modas dominantes (`top`, `freq`) en los textos para detectar redundancias y distribuciones de mercado antes de la normalización.


In [49]:
df.describe(include='all').T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
freelancer_ID,1000,1000,FL250001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
name,1000,992,Lisa Johnson,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,1000,10,FEMALE,115,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,970.0,NaN,NaN,NaN,40.509278,11.942605,20.0,31.0,41.0,51.0,60.0
country,1000,21,South Korea,68,NaN,NaN,NaN,NaN,NaN,NaN,NaN
language,1000,16,English,215,NaN,NaN,NaN,NaN,NaN,NaN,NaN
primary_skill,1000,10,DevOps,112,NaN,NaN,NaN,NaN,NaN,NaN,NaN
years_of_experience,949.0,NaN,NaN,NaN,11.340358,9.68061,0.0,3.0,9.0,17.0,41.0
hourly_rate (USD),906,18,40,94,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rating,899.0,NaN,NaN,NaN,2.51257,1.546599,0.0,1.4,2.6,3.8,5.0


> **Conclusión General del Diagnóstico Estadístico:** El análisis descriptivo global revela un dataset con una distribución demográfica simétrica en edad (media de 40.5 años en un rango estricto de 20 a 60) y una alta fragmentación de mercado liderada por profesionales de DevOps en Corea del Sur e idioma inglés, evidenciando además dos problemas críticos para el negocio: un bajo desempeño promedio en la plataforma (rating de 2.51/5.0) y un severo desorden estructural debido a variables booleanas y numéricas (`gender`, `is_active`, `hourly_rate` y `client_satisfaction`) corrompidas con formatos mixtos, textos redundantes y caracteres especiales (`%`, `$`, `USD`) que obligan a su tratamiento urgente en el módulo de preparación.

## Valores faltantes
Identificamos cuántos valores faltan en cada columna después de la limpieza inicial.


In [52]:
missing = df.isna().sum()
missing[missing > 0].sort_values(ascending=False)


client_satisfaction    176
rating                 101
hourly_rate (USD)       94
is_active               89
years_of_experience     51
age                     30
dtype: int64

## Análisis de variables categóricas
Revisamos las principales frecuencias de `country`, `language`, `primary_skill`, `gender` e `is_active`.


In [51]:
categorical_cols = ['country', 'language', 'primary_skill', 'gender', 'is_active']
for col in categorical_cols:
    print(f'--- {col} ---')
    print(df[col].value_counts(dropna=False).head(10))
    print()


--- country ---
country
South Korea       68
Canada            65
Germany           52
Australia         51
Netherlands       51
United Kingdom    50
Mexico            50
United States     49
China             49
Argentina         47
Name: count, dtype: int64

--- language ---
language
English       215
Spanish       142
Korean         68
German         52
Dutch          51
Mandarin       49
Russian        47
Indonesian     46
Turkish        45
Hindi          45
Name: count, dtype: int64

--- primary_skill ---
primary_skill
DevOps                    112
UI/UX Design              109
Blockchain Development    105
Web Development           104
Mobile Apps               102
AI                        100
Data Analysis              96
Graphic Design             93
Machine Learning           93
Cybersecurity              86
Name: count, dtype: int64

--- gender ---
gender
FEMALE    115
M         106
f         103
Male      103
MALE      102
male      100
m          99
Female     96
F        